In [162]:
import ast
import string

import pandas as pd

DATASET_PATH = "../../data/out/splits/random/gpqa/train_corrected_answer_deepseek_v4_pro.parquet"

df = pd.read_parquet(DATASET_PATH)
print(f"Path: {DATASET_PATH}")
print(f"Shape: {df.shape}")
df.head(2)

Path: ../../data/out/splits/random/gpqa/train_corrected_answer_deepseek_v4_pro.parquet
Shape: (421, 9)


,question,options,answer,answer_index,question_id,distill_ans_correct,distill_reasoning,distill_answer,corrected_reasoning
0,Two quantum states with energies E1 and E2 hav...,"['10^-4 eV', '10^-11 eV', '10^-8 eV\n', '10^-9...",A,0,1,True,"We are given: ""Two quantum states with energie...",a,
1,how many of the following compounds exhibit op...,"['3', '4', '5', '6']",B,1,3,True,Kimi will analyze which compounds are opticall...,b,


## Accuracy


In [163]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 1.0000 (421/421)


## Missing distilled reasoning traces


In [164]:
missing_nan = int(df["distill_answer"].isna().sum())
empty_mask = df["distill_answer"].fillna("").str.strip() == ""
missing_or_empty = int(empty_mask.sum())
print(f"NaN distill_reasoning:            {missing_nan}")
print(f"Missing or empty distill_reasoning: {missing_or_empty}")

NaN distill_reasoning:            0
Missing or empty distill_reasoning: 0


## Reasoning trace length distribution (chars)


In [165]:
existing_reasoning = df.loc[~empty_mask, "distill_reasoning"]
lengths = existing_reasoning.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count     421.000000
mean     3447.954869
std      2612.590035
min       201.000000
25%      1353.000000
50%      2785.000000
75%      4963.000000
max      9960.000000
Name: distill_reasoning, dtype: float64
P60: 3407.0
P70: 4351.0
P80: 5866.0
P85: 6659.0
P90: 7625.0
P92: 8252.800000000001
P95: 8932.0
P99: 9805.8


In [166]:
corrected_empty_mask = df["corrected_reasoning"].fillna("").str.strip() == ""
existing_corrected = df.loc[~corrected_empty_mask, "corrected_reasoning"]
lengths = existing_corrected.str.len()
print(lengths.describe())
print(f"P60: {lengths.quantile(0.6)}")
print(f"P70: {lengths.quantile(0.7)}")
print(f"P80: {lengths.quantile(0.8)}")
print(f"P85: {lengths.quantile(0.85)}")
print(f"P90: {lengths.quantile(0.9)}")
print(f"P92: {lengths.quantile(0.92)}")
print(f"P95: {lengths.quantile(0.95)}")
print(f"P99: {lengths.quantile(0.99)}")

count      25.000000
mean      735.560000
std       218.947878
min       371.000000
25%       554.000000
50%       725.000000
75%       900.000000
max      1199.000000
Name: corrected_reasoning, dtype: float64
P60: 786.1999999999999
P70: 825.5999999999999
P80: 932.8000000000002
P85: 991.6
P90: 1001.8
P92: 1008.6000000000001
P95: 1040.9999999999998
P99: 1163.2399999999998


## Reasoning trace length distribution (tokens)

Token counts per reasoning trace under each of the three model tokenizers we use: `llama3b` (`meta-llama/Llama-3.2-3B-Instruct`), `phi4mini` (`microsoft/Phi-4-mini-instruct`), and `qwen3b` (`Qwen/Qwen2.5-3B-Instruct`).

In [ ]:
# from tqdm.auto import tqdm
# from transformers import AutoTokenizer

# MODEL_TOKENIZERS = {
#     "llama3b": "meta-llama/Llama-3.2-3B-Instruct",
#     "phi4mini": "microsoft/Phi-4-mini-instruct",
#     "qwen3b": "Qwen/Qwen2.5-3B-Instruct",
# }

# reasoning_texts = existing_reasoning.tolist()

# for name, model_id in MODEL_TOKENIZERS.items():
#     tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
#     token_lengths = pd.Series(
#         [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in tqdm(reasoning_texts, desc=name)],
#         index=existing_reasoning.index,
#     )
#     print(f"=== {name} ({model_id}) ===")
#     print(token_lengths.describe())
#     print(f"P60: {token_lengths.quantile(0.6)}")
#     print(f"P70: {token_lengths.quantile(0.7)}")
#     print(f"P80: {token_lengths.quantile(0.8)}")
#     print(f"P85: {token_lengths.quantile(0.85)}")
#     print(f"P90: {token_lengths.quantile(0.9)}")
#     print(f"P92: {token_lengths.quantile(0.92)}")
#     print(f"P95: {token_lengths.quantile(0.95)}")
#     print(f"P99: {token_lengths.quantile(0.99)}")
#     print()

## Invalid answers

An answer is valid iff, after stripping/lowercasing, it is one of the option letters `a`, `b`, ... up to the row's number of options.


In [149]:
LETTERS = list(string.ascii_lowercase)


def n_options(opts):
    if isinstance(opts, str):
        try:
            return len(ast.literal_eval(opts))
        except Exception:
            return 0
    try:
        return len(opts)
    except TypeError:
        return 0


def is_valid_answer(ans, n_opts):
    if not isinstance(ans, str):
        return False
    return ans.strip().lower() in set(LETTERS[:n_opts])


# n_opts_series = df["options"].apply(n_options)
valid_mask = df["distill_ans_correct"]
n_invalid = int((~valid_mask).sum())
print(f"Invalid distill_answer count: {n_invalid} ({n_invalid / n:.2%})")

Invalid distill_answer count: 39 (8.92%)


### All invalid answers


In [150]:
invalid_answers = df.loc[~valid_mask, "distill_reasoning"]
for idx, ans in invalid_answers.items():
    print(f"--- row {idx} ---")
    print(repr(ans))

--- row 4 ---
'Let me analyze this problem step by step.\n\nThe given compound is a benzene ring with five substituents:\n1. -CH₂-CO-CH₃\n2. -CH=CH-CHO\n3. -CH₂-CH=CH-COOH\n4. -CHO\n5. -CH=CH₂\n\nFirst, I need to determine what happens when this compound reacts with excess LiAlH₄.\n\nLiAlH₄ reduces:\n- C=O (aldehydes, ketones, carboxylic acids) → alcohols\n- It does NOT reduce C=C double bonds (isolated alkenes)\n\nLet me go through each substituent:\n\n1. -CH₂-CO-CH₃: The ketone (C=O) gets reduced to -CH₂-CH(OH)-CH₃\n2. -CH=CH-CHO: The aldehyde (C=O) gets reduced to -CH=CH-CH₂OH (the C=C is NOT reduced by LiAlH₄)\n3. -CH₂-CH=CH-COOH: The carboxylic acid gets reduced to -CH₂-CH=CH-CH₂OH (the C=C is NOT reduced)\n4. -CHO: Gets reduced to -CH₂OH\n5. -CH=CH₂: This is just an alkene, NOT reduced by LiAlH₄\n\nNow, the product has:\n- Benzene ring (1 ring + 3 double bonds = DBE of 4)\n- C=C in substituent 2: 1 double bond\n- C=C in substituent 3: 1 double bond\n- C=C in substituent 5: 1 doub

### 10 shortest reasoning traces


In [151]:
shortest = existing_reasoning.str.len().nsmallest(30)
for idx, length in shortest.items():
    print(f"--- row {idx} (len={length}) ---")
    print(df.loc[idx, "distill_reasoning"])
    print()

--- row 55 (len=201) ---
We need answer MCQ single symbol. Need solve. Potential = q/(4pi eps0) 1/sqrt(R^2+z^2) = q/(4pi eps0 R) (1 + (z/R)^2)^(-1/2). Expand = 1 - 1/2 (z/R)^2 + ... = 1 - z^2/(2R^2). Option a. Need output "a".

--- row 72 (len=202) ---
Kimi should respond single letter. Should compute wavelength 1240/2.3393=530.06 nm green emitted. Absorbed complementary color red? Dye emits green, absorption complementary red. Options a. Keep single.

--- row 243 (len=222) ---
Kimi will respond with a single symbol. Should reason. A(Fe)=7.4 dex solar photosphere typically number abundance log N(Fe) on scale log N(H)=12. [Fe/H]=-0.3 target A(Fe)=7.1, H log 12. N_H=10^12. Option c. Respond c only.

--- row 105 (len=224) ---
I'll respond with a single letter. Reaction enamine cyclohexanone pyrrolidine TsOH enamine B. Alkylation methyl bromoacetate gives alpha alkylated iminium/enamine described methyl 2-(2-(pyrrolidin-1-yl)cyclohexyl)acetate. b.

--- row 308 (len=242) ---
Kimi will choos

## Reset distill columns for invalid answers

Sets `distill_reasoning` and `distill_answer` to `""` and `distill_ans_correct` to `False` for rows where the answer is invalid, then writes back to `DATASET_PATH`.


In [140]:
invalid_idx = df.index[~valid_mask]
df.loc[invalid_idx, "distill_reasoning"] = ""
df.loc[invalid_idx, "distill_answer"] = ""
df.loc[invalid_idx, "distill_ans_correct"] = False

df.to_parquet(DATASET_PATH, index=False)
print(f"Reset {len(invalid_idx)} rows and wrote {DATASET_PATH}")

Reset 38 rows and wrote ../../data/out/splits/random/gpqa/train_distilled_deepseek_v4_flash_extend_w_v4_pro_short.parquet


## Reset reasoning traces shorter than 10 chars

Sets `distill_reasoning` to `""` for rows where the existing trace is shorter than 10 characters, then writes back to `DATASET_PATH`.

In [141]:
short_mask = df["distill_reasoning"].fillna("").str.len() < 100
short_idx = df.index[short_mask]
df.loc[short_idx, "distill_reasoning"] = ""
df.loc[short_idx, "distill_answer"] = ""
df.loc[short_idx, "distill_ans_correct"] = False
print(f"distill_reasoning rows under {200} chars: {len(short_idx)}")

# df.to_parquet(DATASET_PATH, index=False)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")

distill_reasoning rows under 200 chars: 38


In [152]:
max_reasoning_chars = 10000
long_mask = df["distill_reasoning"].str.len() > max_reasoning_chars
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
# df.loc[long_mask, "distill_reasoning"] = (
#     df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
# )
df.loc[long_mask, "distill_reasoning"] = ""
df.loc[long_mask, "distill_answer"] = ""
df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 10000 chars: 6


In [153]:
missing_reasoning_mask = df["distill_reasoning"].fillna("").eq("")
print(f"Dropping {int(missing_reasoning_mask.sum())} rows with NA or empty distill_reasoning")
df = df.loc[~missing_reasoning_mask].copy()

Dropping 16 rows with NA or empty distill_reasoning


In [158]:
max_reasoning_chars = 6000
long_mask = (df["distill_reasoning"].str.len() > max_reasoning_chars) & ~valid_mask
print(f"distill_reasoning rows over {max_reasoning_chars} chars: {int(long_mask.sum())}")
df.loc[long_mask, "distill_reasoning"] = (
    df.loc[long_mask, "distill_reasoning"].str.slice(stop=max_reasoning_chars)
)
# df.loc[long_mask, "distill_reasoning"] = ""
# df.loc[long_mask, "distill_answer"] = ""
# df.loc[long_mask, "distill_ans_correct"] = False

distill_reasoning rows over 6000 chars: 0


In [167]:
# max_corrected_reasoning_chars = 1444
# max_combined_reasoning_chars = 8192

# df["corrected_reasoning"] = df["corrected_reasoning"].str.slice(
#     stop=max_corrected_reasoning_chars
# )
has_corrected_reasoning = df["corrected_reasoning"].fillna("").str.strip().ne("")
df.loc[has_corrected_reasoning, "distill_reasoning"] = [
    reasoning + corrected
    for reasoning, corrected in zip(
        df.loc[has_corrected_reasoning, "distill_reasoning"],
        df.loc[has_corrected_reasoning, "corrected_reasoning"],
    )
]

In [8]:
df["distill_reasoning"] = df["distill_answer"]

In [9]:
from core.utils.cot_answers import extract_math_answer

df["distill_answer"] = [extract_math_answer(assistant_response, []) for assistant_response in df["distill_answer"]]


In [168]:
df.head()


,question,options,answer,answer_index,question_id,distill_ans_correct,distill_reasoning,distill_answer,corrected_reasoning
0,Two quantum states with energies E1 and E2 hav...,"['10^-4 eV', '10^-11 eV', '10^-8 eV\n', '10^-9...",A,0,1,True,"We are given: ""Two quantum states with energie...",a,
1,how many of the following compounds exhibit op...,"['3', '4', '5', '6']",B,1,3,True,Kimi will analyze which compounds are opticall...,b,
2,A coating is applied to a substrate resulting ...,"['129°', '124°', '134°', '139°']",B,1,4,True,I'll solve. I'll respond single letter. Should...,b,
3,Consider the following metric:\n\nds^{2}=\frac...,"['0', '+\\infty', '4\\pi\\left(x^{2}-y^{2}\\ri...",B,1,5,True,We need answer multiple choice. Need solve. Th...,b,
4,What is the double bond equivalence of the pro...,"['5', '7', '6', '4']",A,0,7,True,Let me analyze this problem step by step.\n\nT...,a,Wait—I need to reconsider the reduction behavi...


In [169]:
df.describe()

,answer_index
count,421.000000
mean,1.422803
std,1.091898
min,0.000000
25%,0.000000
50%,1.000000
75%,2.000000
max,3.000000


In [170]:
n = len(df)
n_correct = int(df["distill_ans_correct"].sum())
print(f"Accuracy: {n_correct / n:.4f} ({n_correct}/{n})")

Accuracy: 1.0000 (421/421)


In [171]:
df.to_parquet(
    "../../data/out/splits/random/gpqa/train_corrected_answer_deepseek_v4_pro_concat.parquet", index=False
)
# print(f"Reset {len(short_idx)} rows and wrote {DATASET_PATH}")